# USalign Structure Alignment

![USalign Structure Alignment](https://proto-bio.github.io/proto-assets/images/tool/usalign/hero.png)

This notebook demonstrates pairwise structure alignment using USalign. We align two structures provided as PDB text and inspect the TM-scores, which measure topological similarity for both monomers and multi-chain assemblies.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("usalign")
display_overview("usalign")
display_docs_section("usalign", "Background")

# USalign

[USalign](https://github.com/pylelab/USalign) is a universal structure alignment program that extends the TMalign approach to multi-chain complexes and nucleic acid structures. It is developed jointly by the [Zhang Lab](https://zhanggroup.org/) and the [Pyle Lab](https://pylelab.org/) and unifies pairwise structure alignment of proteins, RNA, DNA, and mixed-molecule complexes under a single Template Modeling (TM-score) objective. This toolkit compiles USalign from the canonical [pylelab/USalign](https://github.com/pylelab/USalign) distribution and runs it through a single registered tool that returns both length-normalised TM-scores in oligomeric multi-chain mode.

USalign ([Zhang, Shine, Pyle, and Zhang, 2022](https://doi.org/10.1038/s41592-022-01585-1)) is a universal structure alignment platform that aligns monomer and complex structures of proteins, RNA, and DNA under a single Template Modeling (TM-score) objective. Multi-chain complexes are aligned jointly by combining residue-level structural alignment with a chain-to-chain mapping search, and nucleic acid residues are anchored on the C3' backbone atom in place of the protein Cα. The published benchmark reports consistent advantages over state-of-the-art methods in pairwise and multiple structure alignment across these molecular types, and demonstrates that heterogeneous oligomeric complexes such as protein-RNA assemblies can be aligned within the same framework.

The Template Modeling score (TM-score) ([Zhang and Skolnick, 2004](https://doi.org/10.1002/prot.20264)) is a length-independent measure of topological similarity between two structures. The score lies between 0 and 1, with 1 indicating identical structures. A subsequent statistical analysis ([Xu and Zhang, 2010](https://doi.org/10.1093/bioinformatics/btq066)) established that a TM-score above 0.5 is a strong probabilistic indicator of shared SCOP and CATH fold classification for single-domain proteins, while scores below 0.17 are indistinguishable from random pairs (the random-pair distribution is centred near a TM-score of 0.15). The same 0.5 threshold is the convention used in the literature for multi-chain complex alignment, although the underlying statistical study was performed on monomers.

## Available tools

In [2]:
display_available_tools("usalign")

- **`run_usalign()`** — Universal structure alignment using USalign (Zhang et al., 2022). Supports monomers and multimers. Returns TM-scores normalized by each structure.

### `run_usalign`

USalign performs universal structure alignment that handles monomers, multimeric complexes, nucleic acids, and mixed protein-nucleic acid assemblies. It accepts two structures as raw PDB text, automatically identifies chains and molecular types, computes an optimal chain-to-chain mapping, and returns two TM-scores normalized by the length of each structure. Using it with multimer flags makes it the recommended tool when inputs may contain multiple chains or non-protein molecules.

In [3]:
from pathlib import Path

from proto_tools import Structure, USalignConfig, USalignInput, USalignOutput, run_usalign

In [4]:
# Display input docs
display_api_reference("usalign", "input", "run_usalign")

# Align two homologous TIM-barrel complexes (1TIM, 8TIM) fetched from the RCSB.
# Each is a homodimer, so this also exercises USalign's chain-to-chain mapping.
query = Structure.from_rcsb("1TIM")
reference = Structure.from_rcsb("8TIM")
inputs = USalignInput(query_structure=query, reference_structure=reference)

**Input** — `USalignInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>query_structure</code> | <code>Structure</code> | required | Query / candidate structure (Structure object, file path, or raw PDB/CIF string) |
| <code>reference_structure</code> | <code>Structure</code> | required | Reference / target structure (Structure object, file path, or raw PDB/CIF string) |

In [5]:
# Display config docs
display_api_reference("usalign", "config", "run_usalign")

# USalignConfig has no tool-specific parameters; all defaults are used
config = USalignConfig()

**Config** — `USalignConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cpu'</code> | Device to run the tool on (e.g., 'cpu', 'cuda', 'cuda:0', 'cloud') |
| <code>timeout</code> | <code>int &#124; None</code> | <code>3600</code> | Maximum execution time in seconds. None waits indefinitely. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |

In [6]:
# Run the alignment
result = run_usalign(inputs, config)

Running run_usalign [00:00]

In [7]:
# Display output docs and inspect results
display_api_reference("usalign", "output", "run_usalign")

print(f"TM-score (norm by Structure 1): {result.tm_score_structure_1:.3f}")
print(f"TM-score (norm by Structure 2): {result.tm_score_structure_2:.3f}")
print(f"Execution time: {result.execution_time:.2f}s")

**Output** — `USalignOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>metrics</code> | <code>USalignMetrics</code> | <code>USalignMetrics(primary_metric=None, metric_type='USalignMetrics')</code> | Pairwise alignment scores |

**Metrics** — `USalignMetrics`

| Metric | Type | Range | Unit | Description |
|--------|------|-------|------|-------------|
| <code>tm_score_structure_1</code> | <code>float</code> | <code>[0, 1]</code> |  |  |
| <code>tm_score_structure_2</code> | <code>float</code> | <code>[0, 1]</code> |  |  |

TM-score (norm by Structure 1): 0.987
TM-score (norm by Structure 2): 0.987
Execution time: 1.83s


## Export Results

Alignment results can be exported to JSON format for downstream analysis or integration with other computational pipelines.

In [8]:
output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

result.export("usalign_result", export_path=output_dir, file_format="json")
print(f"Exported to {output_dir / 'usalign_result.json'}")

Exported to example_output/usalign_result.json
